# Checkpoint 5 - Wines: Estatística e K-means
**Disciplinas:** Statistical Computing with R & Python e Machine Learning & Modelling  
**Integrantes:** Maria Eduarda, Giovanna, Athur e Felipe  
**Matrículas:** PREENCHER

Este notebook realiza a análise exploratória de duas variáveis quantitativas e aplica K-means ao dataset Wines.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('wine.csv')
df.head()

## Parte 1 - Statistical Computing with R & Python
Variáveis selecionadas: **Alcohol** e **Malic.acid**.

In [2]:
variaveis = ['Alcohol','Malic.acid']
for col in variaveis:
    print('\n', col)
    print(df[col].describe())
    print('Moda:', df[col].mode().tolist())
    print('Q1, Mediana, Q3:', df[col].quantile([0.25,0.5,0.75]).to_dict())
    print('Variância:', df[col].var())
    print('Desvio padrão:', df[col].std())
    print('Coef. variação (%):', df[col].std()/df[col].mean()*100)


In [3]:
# Tabelas de frequência e histogramas
for col in variaveis:
    freq = pd.cut(df[col], bins=7).value_counts().sort_index()
    display(freq.to_frame('Frequência'))
    df[col].hist(bins=7, edgecolor='black')
    plt.title(f'Distribuição de {col}')
    plt.xlabel(col)
    plt.ylabel('Frequência')
    plt.show()


In [4]:
# Análise probabilística assumindo aproximação Normal
for col in variaveis:
    mu, sigma = df[col].mean(), df[col].std()
    q1, q3 = df[col].quantile([0.25,0.75])
    p_acima_media = 1 - norm.cdf(mu, loc=mu, scale=sigma)
    p_entre_q1_q3 = norm.cdf(q3, loc=mu, scale=sigma) - norm.cdf(q1, loc=mu, scale=sigma)
    print(f'{col}: P(X > média) = {p_acima_media:.4f}')
    print(f'{col}: P(Q1 < X < Q3) = {p_entre_q1_q3:.4f}')


## Parte 2 - Machine Learning & Modelling
A variável **Wine** representa a classe original e não é utilizada no treinamento não supervisionado. As demais features são padronizadas para evitar que escalas diferentes dominem as distâncias do K-means.

In [5]:
X = df.drop(columns=['Wine'])
print('Nulos:', X.isnull().sum().sum())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [6]:
# Elbow e Silhouette para k de 2 a 10
inercias, silhuetas = [], []
ks = range(2,11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_scaled, labels))
    print(k, round(km.inertia_,2), round(silhuetas[-1],4))

plt.plot(list(ks), inercias, marker='o'); plt.title('Método Elbow'); plt.xlabel('k'); plt.ylabel('Inércia'); plt.show()
plt.plot(list(ks), silhuetas, marker='o'); plt.title('Silhouette Score'); plt.xlabel('k'); plt.ylabel('Score'); plt.show()


In [7]:
# Ajuste final com 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cluster = df.copy()
df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)
medias = df_cluster.groupby('Cluster').mean(numeric_only=True)
display(medias[['Alcohol','Malic.acid','Ash','Phenols','Flavanoids','Color.int','Hue','Proline']].round(3))


### Interpretação dos clusters
- **Cluster 0:** menor teor alcoólico médio, menor intensidade de cor e menor Proline; perfil mais leve.
- **Cluster 1:** maior Malic.acid médio e maior intensidade de cor; grupo mais ácido pelo ácido málico.
- **Cluster 2:** maior teor alcoólico médio e Proline muito mais alto; também apresenta valores elevados de flavonoides e fenóis.

### Conclusão
A análise combinada de Elbow e Silhouette indicou **3 clusters** como uma escolha adequada. A padronização foi essencial para equilibrar as escalas e permitir que o K-means diferenciasse perfis químicos de vinhos.